In [45]:
%matplotlib ipympl

In [46]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.signal import find_peaks
import scipy.constants as const

import scqubits as scq


import matplotlib.pyplot as plt
from numpy.linalg import inv, eig


from scipy.signal import find_peaks

# Physical constants
h = const.h
hbar = const.hbar
e = const.e
pi = np.pi
Phi0 = h / (2 * e)


In [47]:
# ================= DESIGN PARAMETERS =================

# Junction geometry
W_um = 0.6
L_um = 2.0
C_area_fF_per_um2 = 45.0

# Plasma frequency target
fp_target_GHz = 30
wp_target = 2 * pi * fp_target_GHz * 1e9

# Derived junction params
Area_um2 = W_um * L_um
CJ = C_area_fF_per_um2 * Area_um2 * 1e-15
LJ = 1.0 / (wp_target**2 * CJ)

EJ = (Phi0 / (2 * pi))**2 / LJ
EJ_GHz = EJ / h / 1e9 * 2

EC = e**2 / (2 * CJ)
EC_GHz = EC / h / 1e9

# Environment
Cg = 20e-18
R = 50.0
d = 0.01

# Evaluation flux
Phi_eval = 0.5 * Phi0

qubit_params = dict(
    EJ_GHz=EJ_GHz,
    EC_GHz=EC_GHz,
    d=d,
    ng=0.0,
    ncut=20,
    truncated_dim=10,
)


In [48]:
# ================= SWEEP RANGES =================

N_list = np.arange(80, 151, 5)

Cc_list = np.logspace(
    np.log10(0.001e-15),
    np.log10(10e-15),
    21
)

print("N sweep:", N_list)
print("Cc sweep:", Cc_list)
print("Total points:", len(N_list) * len(Cc_list))


N sweep: [ 80  85  90  95 100 105 110 115 120 125 130 135 140 145 150]
Cc sweep: [1.00000000e-18 1.58489319e-18 2.51188643e-18 3.98107171e-18
 6.30957344e-18 1.00000000e-17 1.58489319e-17 2.51188643e-17
 3.98107171e-17 6.30957344e-17 1.00000000e-16 1.58489319e-16
 2.51188643e-16 3.98107171e-16 6.30957344e-16 1.00000000e-15
 1.58489319e-15 2.51188643e-15 3.98107171e-15 6.30957344e-15
 1.00000000e-14]
Total points: 315


In [49]:
# ======================================================================
# 1. SQUID + TRANSMON BASICS  (USED FOR PURCELL, NOT FOR MATRIX MODES)
# ======================================================================

def squid_EJ_eff(EJ, d, Phi_ext):
    EJ1 = EJ * (1 + d)
    EJ2 = EJ * (1 - d)
    return np.sqrt(EJ1**2 + EJ2**2 + 2 * EJ1 * EJ2 * np.cos(2*pi*Phi_ext/Phi0))

def squid_LJ(EJ_eff):
    return Phi0**2 / (4 * pi**2 * EJ_eff)

def squid_plasma_freq(EJ_eff, Cq):
    Lj = squid_LJ(EJ_eff)
    return 1.0 / np.sqrt(Lj * Cq)       # [rad/s]


# ======================================================================
# 2. ENVIRONMENT ADMITTANCE (CORRECT NEW MODEL)
# ======================================================================

def Y_feedline(w, Cc, R):
    x = Cc * R * w
    return (Cc**2 * R * w**2 + 1j * Cc * w) / (1 + x**2)

def build_chain_admittance_matrix(N, L_sec, C_sec, Cg, Cc, R, w,
                                  break_middle=True):
    n_nodes = N + 1
    Y = np.zeros((n_nodes, n_nodes), dtype=complex)

    Y_sec = 1j*w*C_sec + 1/(1j*w*L_sec)
    j_mid = N//2
    mid_left, mid_right = j_mid, j_mid + 1

    # LC branches except the middle one (reserved for qubit)
    for j in range(N):
        if break_middle and j == j_mid:
            # Y[i1,i1] += 1j*w*C_sec
            # Y[i2,i2] += 1j*w*C_sec
            # Y[i1,i2] -= 1j*w*C_sec
            # Y[i2,i1] -= 1j*w*C_sec
            continue
        i1, i2 = j, j+1
        Y[i1,i1] += Y_sec
        Y[i2,i2] += Y_sec
        Y[i1,i2] -= Y_sec
        Y[i2,i1] -= Y_sec

    # shunt Cg
    Y_shunt = 1j*w*Cg
    for k in range(n_nodes):
        Y[k,k] += Y_shunt

    # feedline coupling at node 0
    if Cc > 0:
        Y[0,0] += Y_feedline(w, Cc, R)

    return Y, mid_left, mid_right

def Y_env_between_middle_nodes(N, L_sec, C_sec, Cg, Cc, R, w):
    Ymat, iL, iR = build_chain_admittance_matrix(N, L_sec, C_sec, Cg, Cc, R, w)
    n = Ymat.shape[0]
    I = np.zeros(n, dtype=complex)
    I[iL] = 1
    I[iR] = -1
    V = np.linalg.solve(Ymat, I)
    Zenv = V[iL] - V[iR]
    return 1/Zenv

def Y_env_spectrum(N, L_sec, C_sec, Cg, Cc, R, freqs):
    out = []
    for f in freqs:
        w = 2*pi*f
        out.append(Y_env_between_middle_nodes(N, L_sec, C_sec, Cg, Cc, R, w))
    return np.array(out)


# ======================================================================
# 3. PURCELL T1 USING Y_env
# ======================================================================


def transmon_freq_and_phi01(qubit_params, flux):
    """
    Use scqubits TunableTransmon to get:
    - ωq(Φ) from 0→1 transition
    - approximate φ_01(Φ) via sin(phi) operator (small-phase limit)
    """
    qubit = scq.TunableTransmon(
        EJmax=qubit_params['EJ_GHz'],
        EC=qubit_params['EC_GHz'],
        ng=qubit_params['ng'],
        d=qubit_params['d'],
        ncut=qubit_params['ncut'],
        truncated_dim=qubit_params['truncated_dim'],
        flux=flux/Phi0,
    )
    evals, evecs = qubit.eigensys(evals_count=2)
    # evals in GHz; convert to angular frequency
    w01 = 2 * pi * (evals[1] - evals[0]) * 1e9

    # Use sin(phi) operator; in small-phase regime sin φ ≈ φ
    sinphi = qubit.sin_phi_operator()        # matrix in charge basis
    sinphi01 = (evecs[:, 0].conj().T @ sinphi @ evecs[:, 1]).item()
    phi01 = sinphi01  # dimensionless φ_01 ≈ <0|sin φ|1>

    return w01, phi01


def purcell_gamma_T1(CJ, Cq, Yenv):
    ReY = np.real(Yenv)
    if ReY <= 0:
        return 0.0, np.inf
    Gamma = ReY / (2*CJ+Cq)
    return Gamma, 1/Gamma


def purcell_gamma_T1_qub(omega_q, phi01, Yenv):

    prefactor = hbar / (4 * e**2)  # ~1.0e3 in SI units (Ohms)   

    ReY = np.real(Yenv)
    if ReY <= 0:
        return 0.0, np.inf
    Gamma = prefactor * 2 * omega_q * (np.abs(phi01)**2) * ReY
    return Gamma, 1/Gamma

# ======================================================================
# 4. ORIGINAL MATRIX-CHAIN MODEL (FOR MODE SPECTRA ONLY)
# ======================================================================

def build_chain_plain(N, LJ, CJ, Cg, Cc, R):
    n = N + 2
    K = np.zeros((n,n))
    C = np.zeros((n,n))
    G = np.zeros((n,n))

    # JJ inductors & capacitors
    for j in range(1, N+1):
        i1, i2 = j, j+1
        valL = 1/LJ
        K[i1,i1] += valL; K[i2,i2] += valL
        K[i1,i2] -= valL; K[i2,i1] -= valL

        C[i1,i1] += CJ; C[i2,i2] += CJ
        C[i1,i2] -= CJ; C[i2,i1] -= CJ

    # shunt Cg
    for k in range(2, N+2):
        C[k,k] += Cg

    # port Cc
    C[0,0] += Cc; C[1,1] += Cc
    C[0,1] -= Cc; C[1,0] -= Cc

    G[0,0] = 1/R
    return K,C,G


def build_chain_squid(N, LJ, CJ, Cg, Cc, R, qubit_params, Phi_ext, N_SQ=None):
    if N_SQ is None:
        N_SQ = N//2

    w01, _ = transmon_freq_and_phi01(qubit_params, Phi_ext)

    # EJ_eff = squid_EJ_eff(qubit_params['EJ_GHz'], qubit_params['d'], Phi_ext)
    # LJ_s = squid_LJ(EJ_eff)

    CJ_s = 2*CJ
    LJ_s = 1.0 / (w01**2 * CJ_s)

    n = N + 2
    K = np.zeros((n,n))
    C = np.zeros((n,n))
    G = np.zeros((n,n))

    # inductors & capacitors, but one replaced by SQUID
    for j in range(1,N+1):
        i1,i2 = j, j+1
        if j == N_SQ:
            valL = 1/LJ_s
            CJ_eff = CJ_s
        else:
            valL = 1/LJ
            CJ_eff = CJ

        K[i1,i1]+=valL; K[i2,i2]+=valL
        K[i1,i2]-=valL; K[i2,i1]-=valL

        C[i1,i1]+=CJ_eff; C[i2,i2]+=CJ_eff
        C[i1,i2]-=CJ_eff; C[i2,i1]-=CJ_eff

    # shunt Cg
    for k in range(2, N+2):
        C[k,k]+=Cg

    # port Cc
    C[0,0]+=Cc; C[1,1]+=Cc
    C[0,1]-=Cc; C[1,0]-=Cc

    G[0,0]=1/R

    return K,C,G, LJ_s, CJ_s, N_SQ


def s21(K,C,G,R,freqs):
    n = C.shape[0]
    S = []
    b = np.zeros(n); b[0]=1/R
    for f in freqs:
        w = 2*np.pi*f
        M = K - w*w*C + 1j*w*G
        phi = np.linalg.solve(M,b)
        S.append(phi[-1])
    return np.array(S)

def s21_lossy(K, C, G, R, freqs, kappa_i=0, gamma_i=0):
    """
    Adds internal losses to:
        node 1 = resonator:  kappa_i
        node 2 = qubit:      gamma_i
    """

    n = C.shape[0]
    S_phi0 = []
    S_phi1 = []
    S_phi2 = []

    for f in freqs:
        w = 2*np.pi*f

        # copy G each iteration (never modify input)
        G_eff = G.copy()

        # Add internal loss conductances:
        # physically, G has units of siemens (Ohm^-1)
        # damping term is (i*ω)*G in the equations.
        G_eff[1,1] += kappa_i / w       # resonator intrinsic loss
        G_eff[N//2,N//2] += gamma_i / w       # qubit intrinsic loss

        # Standard solve
        b = np.zeros(n)
        b[0] = 1/R

        M = K - w*w*C + 1j*w*G_eff
        phi = np.linalg.solve(M, b)
        S_phi0.append(phi[0])  # measure at port node
        S_phi1.append(phi[1])  # measure at resonator node
        S_phi2.append(phi[N//2])  # measure at resonator node (or phi[-1] if you want qubit node)

    return np.array(S_phi0), np.array(S_phi1), np.array(S_phi2)


def build_A(K,C,G):
    n = C.shape[0]
    Cinv = inv(C)
    zero = np.zeros((n,n))
    I = np.eye(n)
    A = np.block([[zero, I], [-Cinv@K, -Cinv@G]])
    return A

def compute_modes(A):
    lam, vec = eig(A)
    mask = np.imag(lam) > 0
    lam = lam[mask]
    vec = vec[:,mask]
    order = np.argsort(np.imag(lam))
    lam = lam[order]
    vec = vec[:,order]

    omega = np.imag(lam)
    freqs = omega/(2*pi)
    Q = omega / (2*(-np.real(lam)))
    return freqs, Q, lam, vec


def compare_modes(freqs_plain, freqs_squid, f_q):
    plt.figure(figsize=(6,4))
    plt.plot(freqs_plain/1e9, '.', label='Plain chain')
    plt.plot(freqs_squid/1e9, 'x', label='Chain + SQUID')
    plt.axhline(f_q/1e9, color='r', linestyle='--', label='Bare qubit freq')
    plt.xlabel("Mode index")
    plt.ylabel("Frequency [GHz]")
    plt.legend()
    plt.title("Mode Frequencies (Plain vs SQUID)")
    plt.tight_layout()
    plt.show()

In [50]:
results = []


for N in tqdm(N_list, desc="Sweeping N"):
    for Cc in tqdm(Cc_list, desc="Sweeping Cc"):

       

        ###############################################################3


        # ======================================================================
        # 5.3 QUBIT FREQUENCY & PURCELL T1(Φ)
        # ======================================================================


        
        w01, phi01 = transmon_freq_and_phi01(qubit_params, Phi_eval)

        f_q = w01/(2*pi)
        Yenv_q = Y_env_between_middle_nodes(N, LJ, CJ, Cg, Cc, R, w01)
        _, T1_qub = purcell_gamma_T1_qub(w01, phi01, Yenv_q)

        T1_qub = np.clip(T1_qub, 0, 1e3)   # clip to max 1e6 s for plotting
        
    




        Phi_list = np.linspace(0, 0.5*Phi0, 1001)
        f_span = np.linspace(3e9, 31e9, 2500)


        S_phi0 = np.zeros((len(Phi_list), len(f_span)), dtype=complex)
        S_phi1 = np.zeros((len(Phi_list), len(f_span)), dtype=complex)
        S_phi2 = np.zeros((len(Phi_list), len(f_span)), dtype=complex)

        f_q_list_scq = []

        for i, Phi in enumerate(Phi_list):

            w01, phi01 = transmon_freq_and_phi01(qubit_params, Phi)
            f_q_list_scq.append(w01/(2*pi))


            Ks,Cs,Gs, LJ_s, CJ_s, idx_squid = build_chain_squid(
                N, LJ, CJ, Cg, Cc, R, qubit_params, Phi
            )

            kappa_i = 0#2*np.pi*1e3
            gamma_i = 0#2*np.pi*2e6

            S_phi0[i,:], S_phi1[i,:], S_phi2[i,:] = s21_lossy(
                Ks, Cs, Gs,
                R,
                f_span,
                kappa_i=kappa_i,
                gamma_i=gamma_i
            )
        f_q_list_scq = np.array(f_q_list_scq, dtype=float)





        # ###############################################################################
        # # 1) Bare first chain mode (ignore spurious near 0 Hz)
        # ###############################################################################

        # # Rebuild plain chain and its modes (fast; keeps everything self-contained)
        Ks,Cs,Gs, LJ_s, CJ_s, idx_squid = build_chain_squid(
            N, LJ, CJ, Cg, Cc, R, qubit_params, Phi_eval
        )
        As = build_A(Ks,Cs,Gs)
        freq_squid, Q_squid, _, _ = compute_modes(As)

        freq_plain = freq_squid  # use modes with SQUID for better accuracy

        # Ignore spurious ultra-low mode, keep first mode above some threshold
        f_min_th = 1e9   # 1 GHz threshold; adjust if needed
        mask_valid = freq_plain > f_min_th
        if not np.any(mask_valid):
            raise RuntimeError("No chain mode found above f_min_th; adjust threshold.")

        f_chain0 = freq_plain[mask_valid][1]   # bare 0th chain mode [Hz]
        f_chain1 = freq_plain[mask_valid][2]   # bare 1st chain mode [Hz]


        ###############################################################################
        # 2) Flux where qubit (scqubits) crosses the bare chain mode
        ###############################################################################

        # f_q_list_scq is already in Hz from your loop
        diff = np.abs(f_q_list_scq - f_chain0)
        idx_cross = int(np.argmin(diff))          # index in Phi_list where crossing is closest
        if not (Phi_list.shape == f_q_list_scq.shape):
            raise RuntimeError("Phi_list and f_q_list_scq shapes do not match.")
        Phi_cross = Phi_list[idx_cross]           # [Wb]
        Phi_cross_over_Phi0 = Phi_cross / Phi0
        f_q_cross = f_q_list_scq[idx_cross]       # [Hz]

        ###############################################################################
        # 3) On this flux slice, find upper/lower hybridized peaks of the first mode
        ###############################################################################

        # Take |S21| at node 2 (S_phi2) at this flux
        S_line = np.abs(S_phi2[idx_cross, :])   # shape [len(f_span)]

        # Focus on a frequency window around the bare chain mode
        window_half_width = 2e9  # 2 GHz window on each side; adjust if needed
        f_low = f_chain0 - window_half_width
        f_high = f_chain0 + window_half_width

        freq_mask = (f_span >= f_low) & (f_span <= f_high)
        freq_idx = np.where(freq_mask)[0]

        if len(freq_idx) < 10:
            raise RuntimeError("Window around chain mode too small or out of f_span; adjust window.")

        S_win = S_line[freq_mask]
        f_win = f_span[freq_mask]

        # Find peaks in this window
        peaks, properties = find_peaks(S_win, height=np.max(S_win)*0.1)  # 10% height threshold
        if len(peaks) < 2:
            # fallback: take the two highest points as 'peaks'
            sort_idx = np.argsort(S_win)[-2:]
            peaks = np.sort(sort_idx)
        else:
            # take two highest peaks among those found
            peak_heights = S_win[peaks]
            best_two = np.argsort(peak_heights)[-2:]   # indices in 'peaks'
            peaks = np.sort(peaks[best_two])

        # Frequencies of the two hybridized peaks
        f_minus = f_win[peaks[0]]   # lower branch [Hz]
        f_plus  = f_win[peaks[1]]   # upper branch [Hz]

        # Splitting and coupling
        delta_f = f_plus - f_minus      # total splitting [Hz]
        g_Hz = delta_f / 2.0            # g in Hz
        g_rad_s = 2 * np.pi * g_Hz      # g in rad/s
        g_over_2pi_MHz = g_Hz / 1e6     # g/2π in MHz


        ###############################################################



        results.append(dict(
            N=N,
            Cc_F=Cc,
            Cg_F=Cg,
            W_um=W_um,
            L_um=L_um,
            fp_target_GHz=fp_target_GHz,

            R_Ohm=R,
            EJ_GHz=EJ_GHz,
            EC_GHz=EC_GHz,
            d=d,
            f_chain0_GHz=f_chain0/1e9,
            f_chain1_GHz=f_chain1/1e9,
            f_minus_GHz=f_minus/1e9,
            f_plus_GHz=f_plus/1e9,
            g_MHz=g_Hz/1e6,
            T1_Purcell_s=T1_qub,
            f_qubit_GHz=f_q/1e9
        ))

        df = pd.DataFrame(results)
        df.to_pickle("sweep_N_Cc_g_Purcell.pkl")
        df.to_csv("sweep_N_Cc_g_Purcell.csv", index=False)



Sweeping N: 100%|██████████| 15/15 [37:56:13<00:00, 9104.93s/it]   


In [51]:
df = pd.DataFrame(results)

print("Total valid points:", len(df))
df.head()


Total valid points: 315


,N,Cc_F,Cg_F,W_um,L_um,fp_target_GHz,R_Ohm,EJ_GHz,EC_GHz,d,f_chain0_GHz,f_chain1_GHz,f_minus_GHz,f_plus_GHz,g_MHz,T1_Purcell_s,f_qubit_GHz
0,80,1.000000e-18,2.000000e-17,0.6,2.0,30,50.0,627.251221,0.358708,0.01,27.008050,29.136665,26.663866,27.302521,319.327731,11.118576,3.849261
1,80,1.584893e-18,2.000000e-17,0.6,2.0,30,50.0,627.251221,0.358708,0.01,27.006163,29.136058,26.652661,27.302521,324.929972,4.429537,3.849261
2,80,2.511886e-18,2.000000e-17,0.6,2.0,30,50.0,627.251221,0.358708,0.01,27.003171,29.135096,26.652661,27.302521,324.929972,1.765421,3.849261
3,80,3.981072e-18,2.000000e-17,0.6,2.0,30,50.0,627.251221,0.358708,0.01,26.998428,29.133569,26.652661,27.291317,319.327731,0.704083,3.849261
4,80,6.309573e-18,2.000000e-17,0.6,2.0,30,50.0,627.251221,0.358708,0.01,26.990905,29.131147,26.652661,27.291317,319.327731,0.281094,3.849261


In [ ]:
df.to_pickle("sweep_N_Cc_g_Purcell.pkl")
df.to_csv("sweep_N_Cc_g_Purcell.csv", index=False)


: 